# 📨 Day 04a: Inter-Process Communication (IPC)

---

## 🎯 What You'll Master Today
- Shared memory vs message passing
- Pipes (named/unnamed), queues, sockets, signals
- Python IPC: multiprocessing.Pipe, Queue, shared memory

---

## 🎭 The Analogy

**Shared memory** = whiteboard in a shared office (both can read/write — fast but needs coordination)  
**Message passing** = sending letters (slower but no conflict — each has own copy)  
**Pipe** = a pneumatic tube between two rooms (one-way or two-way)  

---

```
╔═════════════════════════════════════════════════════════╗
║              IPC MECHANISMS OVERVIEW                    ║
╠════════════════════╦════════════════════════════════════╣
║  Mechanism          ║  Speed / Notes                   ║
╠════════════════════╬════════════════════════════════════╣
║  Shared Memory      ║  Fastest. Needs sync (locks)     ║
║  Pipes              ║  Unidirectional, parent-child    ║
║  Named Pipes (FIFO) ║  Bidi, unrelated processes       ║
║  Message Queues     ║  Structured messages, OS managed ║
║  Sockets            ║  Network-capable, most flexible  ║
║  Signals            ║  Notifications only (SIGKILL etc)║
╚════════════════════╩════════════════════════════════════╝
```

In [ ]:
# ============================================
# 1. Pipe — Parent-Child Communication
# ============================================
from multiprocessing import Process, Pipe

def child_process(conn):
    conn.send("Hello from child! 👋")
    response = conn.recv()
    print(f"  Child received: {response}")
    conn.close()

# Create pipe — returns two connection objects
parent_conn, child_conn = Pipe()

p = Process(target=child_process, args=(child_conn,))
p.start()

# Parent side
msg = parent_conn.recv()
print(f"  Parent received: {msg}")
parent_conn.send("Parent says hi back! 👋")

p.join()

In [ ]:
# ============================================
# 2. Queue — Producer-Consumer IPC
# ============================================
from multiprocessing import Process, Queue
import time

def producer(q: Queue, items: list):
    for item in items:
        q.put(item)
        print(f"  📦 Produced: {item}")
    q.put(None)  # sentinel — signals "done"

def consumer(q: Queue, name: str):
    while True:
        item = q.get()
        if item is None:
            break
        print(f"  🛒 {name} consumed: {item}")


q = Queue()
prod = Process(target=producer, args=(q, ["task_1", "task_2", "task_3"]))
cons = Process(target=consumer, args=(q, "Worker"))

prod.start()
cons.start()
prod.join()
cons.join()
print("  Done!")

In [ ]:
# ============================================
# 3. Shared Memory — Fastest IPC
# ============================================
from multiprocessing import Process, Value, Array

def increment_shared(counter, lock):
    for _ in range(100_000):
        with lock:
            counter.value += 1

from multiprocessing import Lock

counter = Value('i', 0)  # shared integer
lock = Lock()

procs = [Process(target=increment_shared, args=(counter, lock)) for _ in range(4)]
for p in procs: p.start()
for p in procs: p.join()

print(f"  Shared counter: {counter.value}  (expected: 400,000)")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Shared memory vs message passing? | SM: fast, needs sync. MP: slower, no sync needed, copy semantics |
| 2 | Named pipe vs unnamed pipe? | Unnamed: parent-child only. Named (FIFO): any two processes, persists in filesystem |
| 3 | When to use sockets for IPC? | Cross-machine communication, language-agnostic, most flexible but slowest |
| 4 | What are signals? | Async notifications (SIGTERM, SIGKILL, SIGUSR1). Can't carry data, just notify |
| 5 | How do modern systems do IPC? | Message queues (RabbitMQ, Kafka), shared memory (Redis), RPC (gRPC), Unix domain sockets |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Shared memory = fastest, needs synchronization        │
## │  • Pipes = simple parent-child channel                   │
## │  • Queue = structured message passing (producer-consumer)│
## │  • Sockets = network-capable, most flexible              │
## │  • Signals = lightweight async notifications             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Critical Section & Race Conditions** — the synchronization problem